# FarmaStock AI — MVP con Gemini, RAG, ChromaDB, LangGraph y memoria

Este notebook implementa el MVP de **FarmaStock AI**, un asistente experto en optimización de stock en farmacia comunitaria.

El agente estará especializado en responder preguntas sobre:

- Rotación de stock.
- Cobertura de stock.
- Stock mínimo, stock máximo y stock de seguridad.
- Punto de pedido.
- Demanda histórica.
- Lead time o plazo de reposición.
- Riesgo de rotura.
- Sobrestock.
- Clasificación ABC/XYZ.
- Interpretación de movimientos de inventario.

El objetivo del notebook es construir progresivamente el sistema completo:

1. Cargar la base documental propia.
2. Extraer metadatos YAML y secciones Markdown.
3. Limpiar y segmentar los documentos.
4. Crear chunks con metadatos.
5. Preparar la base para embeddings, ChromaDB, RAG, LangGraph y memoria conversacional.

En esta primera parte del notebook se implementa únicamente el procesamiento documental hasta el chunking.  
La parte de Gemini, ChromaDB, LangGraph y memoria se añadirá después.

FarmaStock AI tiene un enfoque técnico-formativo y logístico. No proporciona consejo clínico, no recomienda tratamientos y no utiliza datos reales sensibles.

## 1. Arquitectura del MVP

La arquitectura prevista para FarmaStock AI sigue este flujo:

```text
Documentos Markdown propios
        ↓
Carga documental
        ↓
Extracción de metadatos YAML
        ↓
Extracción de secciones Markdown
        ↓
Limpieza básica del texto
        ↓
Chunking con metadatos
        ↓
Gemini Embeddings
        ↓
ChromaDB
        ↓
Retriever
        ↓
Agente LangGraph
        ↓
Memoria conversacional
        ↓
Respuesta generada con Gemini
        ↓
Interacción en notebook
```

En este bloque del notebook se implementan las primeras fases:

```text
Documentos Markdown
        ↓
Carga documental
        ↓
YAML
        ↓
Secciones
        ↓
Limpieza
        ↓
Chunks con metadatos
```

Esta preparación es importante porque la calidad del RAG dependerá en gran parte de cómo estén segmentados e indexados los documentos.

In [63]:
PROJECT_NAME = "FarmaStock AI"
PROJECT_DOMAIN = "Optimización de stock en farmacia comunitaria"

print(f"Proyecto: {PROJECT_NAME}")
print(f"Dominio: {PROJECT_DOMAIN}")

Proyecto: FarmaStock AI
Dominio: Optimización de stock en farmacia comunitaria


## 2. Instalación e importación de dependencias

Este notebook necesita varias librerías para cargar documentos, extraer metadatos, dividir texto y preparar los chunks.

En esta primera parte se usan principalmente:

- `pathlib` para gestionar rutas.
- `re` para expresiones regulares.
- `yaml` para leer metadatos YAML.
- `langchain-core` para crear objetos `Document`.
- `langchain-text-splitters` para dividir el texto en chunks.

Más adelante se utilizarán también:

- `langchain-google-genai` para Gemini y Gemini Embeddings.
- `langchain-chroma` y `chromadb` para la base vectorial.
- `langgraph` para construir el agente.
- `python-dotenv` para cargar la API key desde `.env`.

La instalación puede ejecutarse si el entorno aún no tiene las dependencias.

In [64]:
# Ejecutar esta celda solo si el entorno no tiene instaladas las dependencias.
# En Google Colab o un entorno limpio, descomenta la línea siguiente:

# !pip install -q langchain langchain-core langchain-community langchain-google-genai langchain-chroma langchain-text-splitters langgraph chromadb python-dotenv pyyaml pandas

In [65]:
from pathlib import Path
import os
import re
import yaml
from typing import Dict, Any, List, Tuple

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Dependencias importadas correctamente.")

Dependencias importadas correctamente.


## 3. Configuración de entorno y API key

Aunque en este primer bloque todavía no se llama a Gemini, dejamos preparada la configuración de entorno.

La API key de Gemini no debe escribirse directamente en el notebook.  
Debe guardarse en una variable de entorno o en un archivo `.env`.

Ejemplo de archivo `.env` en la raíz del proyecto:

```text
GOOGLE_API_KEY=tu_api_key_de_gemini
```

En esta celda se comprueba si la variable existe.  
No se detiene todavía el notebook si falta, porque en esta primera parte solo estamos preparando documentos y chunks.

In [66]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if GOOGLE_API_KEY:
    print("GOOGLE_API_KEY detectada correctamente.")
else:
    print("Aviso: GOOGLE_API_KEY no encontrada. Será necesaria en el siguiente bloque para Gemini y embeddings.")

GOOGLE_API_KEY detectada correctamente.


## 4. Definición de rutas del proyecto

El notebook espera una estructura de carpetas como esta:

```text
farmastock-ai/
│
├── data/
│   └── raw/
│       ├── 01_fundamentos_stock_farmacia.md
│       ├── 02_metricas_reposicion_farmacia.md
│       ├── 03_clasificacion_abc_xyz_farmacia.md
│       └── 04_interpretacion_movimientos_stock.md
│
├── chroma_db/
│
└── notebooks/
    └── 01_farmastock_ai_mvp.ipynb
```

La variable `BASE_DIR` se calcula de forma que el notebook funcione tanto si se ejecuta desde la carpeta raíz como desde la carpeta `notebooks`.

In [67]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    BASE_DIR = current_dir.parent
else:
    BASE_DIR = current_dir

DATA_DIR = BASE_DIR / "data" / "raw"
CHROMA_DIR = BASE_DIR / "chroma_db"

EXPECTED_FILES = [
    "01_fundamentos_stock_farmacia.md",
    "02_metricas_reposicion_farmacia.md",
    "03_clasificacion_abc_xyz_farmacia.md",
    "04_interpretacion_movimientos_stock.md",
]

print(f"Directorio actual: {current_dir}")
print(f"Directorio base del proyecto: {BASE_DIR}")
print(f"Directorio de documentos: {DATA_DIR}")
print(f"Directorio previsto para ChromaDB: {CHROMA_DIR}")

Directorio actual: c:\Users\alvar\farmastock-ai\notebooks
Directorio base del proyecto: c:\Users\alvar\farmastock-ai
Directorio de documentos: c:\Users\alvar\farmastock-ai\data\raw
Directorio previsto para ChromaDB: c:\Users\alvar\farmastock-ai\chroma_db


In [68]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"No existe el directorio {DATA_DIR}. "
        "Comprueba que la carpeta data/raw/ está creada en la raíz del proyecto."
    )

document_paths = [DATA_DIR / filename for filename in EXPECTED_FILES]

missing_files = [path.name for path in document_paths if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Faltan documentos obligatorios en data/raw/:\n"
        + "\n".join(f"- {filename}" for filename in missing_files)
    )

print("Se han encontrado los 4 documentos obligatorios:")

for path in document_paths:
    print(f"- {path.name}")

Se han encontrado los 4 documentos obligatorios:
- 01_fundamentos_stock_farmacia.md
- 02_metricas_reposicion_farmacia.md
- 03_clasificacion_abc_xyz_farmacia.md
- 04_interpretacion_movimientos_stock.md


## 5. Carga de documentos Markdown

En esta sección se cargan los cuatro documentos Markdown de la base de conocimiento.

Cada documento se guarda inicialmente como texto bruto junto con su ruta y nombre de archivo.  
Todavía no se aplica chunking ni embeddings.

In [69]:
def read_markdown_file(path: Path) -> str:
    """
    Lee un archivo Markdown usando codificación UTF-8.
    """
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    return path.read_text(encoding="utf-8")

In [70]:
raw_docs: List[Dict[str, Any]] = []

for path in document_paths:
    raw_text = read_markdown_file(path)

    raw_docs.append(
        {
            "source": str(path),
            "filename": path.name,
            "raw_text": raw_text,
        }
    )

print(f"Documentos cargados: {len(raw_docs)}")

for doc in raw_docs:
    print(f"- {doc['filename']} | caracteres: {len(doc['raw_text'])}")

Documentos cargados: 4
- 01_fundamentos_stock_farmacia.md | caracteres: 18608
- 02_metricas_reposicion_farmacia.md | caracteres: 19716
- 03_clasificacion_abc_xyz_farmacia.md | caracteres: 20658
- 04_interpretacion_movimientos_stock.md | caracteres: 24288


In [71]:
# Vista rápida del inicio del primer documento
print(raw_docs[0]["raw_text"][:700])

---
title: "Fundamentos de gestión de stock en farmacia comunitaria"
document_id: "01_fundamentos_stock_farmacia"
version: "1.0"
domain: "Optimización de stock en farmacia comunitaria"
use_in_rag: true
contains_real_data: false
---

# Fundamentos de gestión de stock en farmacia comunitaria

## 1. Introducción a la gestión de stock en farmacia comunitaria

La gestión de stock en farmacia comunitaria consiste en controlar, revisar y ajustar las existencias de productos disponibles para que la farmacia pueda responder a la demanda habitual sin acumular un exceso innecesario de inventario. El objetivo no es tener muchas unidades de todos los productos, sino disponer de una cantidad razonable, eq


## 6. Extracción de metadatos YAML

Cada documento comienza con una cabecera YAML que contiene metadatos útiles para trazabilidad y filtrado dentro del sistema RAG.

Metadatos esperados:

- `title`
- `document_id`
- `version`
- `domain`
- `use_in_rag`
- `contains_real_data`

Estos metadatos se conservarán y se propagarán a cada sección y a cada chunk.

In [72]:
REQUIRED_YAML_KEYS = [
    "title",
    "document_id",
    "version",
    "domain",
    "use_in_rag",
    "contains_real_data",
]


def extract_yaml_metadata(text: str) -> Tuple[Dict[str, Any], str]:
    """
    Extrae la cabecera YAML inicial de un documento Markdown.

    Devuelve:
    - metadata: diccionario con los metadatos YAML.
    - body: texto Markdown sin la cabecera YAML.
    """
    if not text.startswith("---"):
        raise ValueError("El documento no empieza con una cabecera YAML delimitada por ---.")

    parts = text.split("---", 2)

    if len(parts) < 3:
        raise ValueError("No se ha podido cerrar correctamente la cabecera YAML.")

    yaml_text = parts[1].strip()
    body = parts[2].strip()

    metadata = yaml.safe_load(yaml_text)

    if metadata is None:
        metadata = {}

    missing_keys = [key for key in REQUIRED_YAML_KEYS if key not in metadata]

    if missing_keys:
        raise ValueError(
            "Faltan claves obligatorias en la cabecera YAML: "
            + ", ".join(missing_keys)
        )

    return metadata, body

In [73]:
parsed_docs: List[Dict[str, Any]] = []

for item in raw_docs:
    metadata, body = extract_yaml_metadata(item["raw_text"])

    metadata["source"] = item["source"]
    metadata["filename"] = item["filename"]

    parsed_docs.append(
        {
            "metadata": metadata,
            "body": body,
        }
    )

print(f"Documentos parseados: {len(parsed_docs)}")

for doc in parsed_docs:
    meta = doc["metadata"]
    print(
        f"- {meta['document_id']} | "
        f"{meta['title']} | "
        f"use_in_rag={meta['use_in_rag']} | "
        f"contains_real_data={meta['contains_real_data']}"
    )

Documentos parseados: 4
- 01_fundamentos_stock_farmacia | Fundamentos de gestión de stock en farmacia comunitaria | use_in_rag=True | contains_real_data=False
- 02_metricas_reposicion_farmacia | Métricas de reposición y criterios de revisión en farmacia comunitaria | use_in_rag=True | contains_real_data=False
- 03_clasificacion_abc_xyz_farmacia | Clasificación ABC/XYZ aplicada a la gestión de stock en farmacia comunitaria | use_in_rag=True | contains_real_data=False
- 04_interpretacion_movimientos_stock | Interpretación de movimientos de stock en farmacia comunitaria | use_in_rag=True | contains_real_data=False


In [74]:
# Validación adicional: todos los documentos deben pertenecer al dominio del proyecto
for doc in parsed_docs:
    domain = doc["metadata"].get("domain")

    if domain != PROJECT_DOMAIN:
        raise ValueError(
            f"El documento {doc['metadata'].get('document_id')} tiene un dominio inesperado: {domain}"
        )

print("Todos los documentos pertenecen al dominio esperado.")

Todos los documentos pertenecen al dominio esperado.


## 7. Extracción de secciones Markdown

Los documentos están redactados con encabezados de nivel 2 (`##`) para facilitar la segmentación semántica.

En esta sección se divide cada documento en secciones principales.

Cada sección conservará:

- Metadatos del documento.
- Número de sección.
- Título de sección.
- Texto de la sección.

Esto permite que cada chunk tenga trazabilidad clara hacia el documento y la sección de origen.

In [75]:
def split_markdown_sections(body: str) -> List[Dict[str, str]]:
    """
    Divide un documento Markdown en secciones usando encabezados de nivel 2.

    Devuelve una lista de diccionarios con:
    - section_number
    - section_title
    - section_text
    """
    pattern = r"(?m)^##\s+(.+)$"
    matches = list(re.finditer(pattern, body))

    if not matches:
        raise ValueError("No se encontraron secciones con encabezado ## en el documento.")

    sections = []

    for index, match in enumerate(matches):
        section_title = match.group(1).strip()
        start = match.start()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(body)

        section_text = body[start:end].strip()

        number_match = re.match(r"^(\d+)\.", section_title)
        section_number = number_match.group(1) if number_match else str(index + 1)

        sections.append(
            {
                "section_number": section_number,
                "section_title": section_title,
                "section_text": section_text,
            }
        )

    return sections

In [76]:
sections: List[Dict[str, Any]] = []

for doc in parsed_docs:
    doc_metadata = doc["metadata"]
    doc_sections = split_markdown_sections(doc["body"])

    for section in doc_sections:
        section_title_lower = section["section_title"].lower()

        # Las secciones finales de preguntas se conservan en los documentos,
        # pero no se indexan en ChromaDB para evitar recuperaciones literales
        # poco informativas durante el RAG.
        if "preguntas que puede responder" in section_title_lower:
            continue

        section_metadata = {
            **doc_metadata,
            "section_number": section["section_number"],
            "section_title": section["section_title"],
        }

        sections.append(
            {
                "text": section["section_text"],
                "metadata": section_metadata,
            }
        )

print(f"Secciones extraídas para indexación: {len(sections)}")

for section in sections[:8]:
    print(
        f"- {section['metadata']['document_id']} | "
        f"Sección {section['metadata']['section_number']}: "
        f"{section['metadata']['section_title']}"
    )

Secciones extraídas para indexación: 41
- 01_fundamentos_stock_farmacia | Sección 1: 1. Introducción a la gestión de stock en farmacia comunitaria
- 01_fundamentos_stock_farmacia | Sección 2: 2. Tipos de stock
- 01_fundamentos_stock_farmacia | Sección 3: 3. Rotación de stock
- 01_fundamentos_stock_farmacia | Sección 4: 4. Cobertura de stock
- 01_fundamentos_stock_farmacia | Sección 5: 5. Stock mínimo, stock máximo y stock de seguridad
- 01_fundamentos_stock_farmacia | Sección 6: 6. Punto de pedido
- 01_fundamentos_stock_farmacia | Sección 7: 7. Roturas de stock
- 01_fundamentos_stock_farmacia | Sección 8: 8. Sobrestock


In [77]:
# Resumen de secciones por documento
sections_by_doc: Dict[str, int] = {}

for section in sections:
    doc_id = section["metadata"]["document_id"]
    sections_by_doc[doc_id] = sections_by_doc.get(doc_id, 0) + 1

for doc_id, count in sections_by_doc.items():
    print(f"{doc_id}: {count} secciones")

01_fundamentos_stock_farmacia: 9 secciones
02_metricas_reposicion_farmacia: 10 secciones
03_clasificacion_abc_xyz_farmacia: 10 secciones
04_interpretacion_movimientos_stock: 12 secciones


## 8. Limpieza básica del texto

Los documentos ya están redactados específicamente para el sistema RAG, por lo que no se aplica una limpieza agresiva.

La limpieza se limita a:

- Normalizar saltos de línea.
- Eliminar espacios repetidos.
- Quitar espacios al inicio y al final.
- Mantener títulos, ejemplos, tablas y fórmulas conceptuales.

No se eliminan encabezados ni bloques de código porque aportan contexto útil para la recuperación.

In [78]:
def clean_text(text: str) -> str:
    """
    Aplica limpieza básica sin destruir la estructura Markdown.
    """
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Elimina espacios y tabulaciones repetidas, sin tocar saltos de línea.
    text = re.sub(r"[ \t]+", " ", text)

    # Reduce bloques excesivos de líneas en blanco.
    text = re.sub(r"\n{4,}", "\n\n\n", text)

    return text.strip()

In [79]:
for section in sections:
    section["text"] = clean_text(section["text"])

print("Limpieza básica aplicada a todas las secciones.")

Limpieza básica aplicada a todas las secciones.


In [80]:
# Vista de ejemplo de una sección limpia
example_section = sections[0]

print("Documento:", example_section["metadata"]["document_id"])
print("Sección:", example_section["metadata"]["section_title"])
print("-" * 80)
print(example_section["text"][:1000])

Documento: 01_fundamentos_stock_farmacia
Sección: 1. Introducción a la gestión de stock en farmacia comunitaria
--------------------------------------------------------------------------------
## 1. Introducción a la gestión de stock en farmacia comunitaria

La gestión de stock en farmacia comunitaria consiste en controlar, revisar y ajustar las existencias de productos disponibles para que la farmacia pueda responder a la demanda habitual sin acumular un exceso innecesario de inventario. El objetivo no es tener muchas unidades de todos los productos, sino disponer de una cantidad razonable, equilibrada y adaptada al comportamiento de cada referencia.

En una farmacia comunitaria conviven productos con comportamientos muy distintos. Algunos tienen una salida frecuente y previsible, otros se venden de forma ocasional, otros dependen de campañas estacionales y otros pueden permanecer mucho tiempo sin movimiento. Por eso, una gestión correcta del stock debe tener en cuenta la rotación, la

## 9. Chunking con metadatos

En esta sección se divide cada sección Markdown en fragmentos más pequeños o chunks.

La estrategia elegida es:

```text
chunk_size = 1000
chunk_overlap = 150
```

Esta configuración busca mantener juntos:

- Definición del concepto.
- Explicación aplicada.
- Ejemplo sencillo.
- Advertencia o límite de uso.

Además, cada chunk conserva metadatos del documento, de la sección y del propio fragmento.

Esto permitirá que, más adelante, ChromaDB recupere fragmentos relevantes y que el agente pueda mostrar o rastrear la fuente de cada respuesta.

In [81]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

print(f"Splitter creado con chunk_size={CHUNK_SIZE} y chunk_overlap={CHUNK_OVERLAP}.")

Splitter creado con chunk_size=1000 y chunk_overlap=150.


In [82]:
def build_chunk_text(section_text: str, metadata: Dict[str, Any]) -> str:
    """
    Añade encabezado contextual al texto antes de dividirlo.
    Esto ayuda al retriever a conservar información de documento y sección.
    """
    document_title = metadata.get("title", "Documento sin título")
    section_title = metadata.get("section_title", "Sección sin título")

    return (
        f"Documento: {document_title}\n"
        f"Sección: {section_title}\n\n"
        f"{section_text}"
    )

In [83]:
chunks: List[Document] = []

for section in sections:
    section_text = section["text"]
    section_metadata = section["metadata"]

    text_with_context = build_chunk_text(section_text, section_metadata)
    split_texts = text_splitter.split_text(text_with_context)

    for chunk_index, chunk_text in enumerate(split_texts):
        document_id = section_metadata["document_id"]
        section_number = section_metadata["section_number"]

        chunk_metadata = {
            **section_metadata,
            "chunk_index": chunk_index,
            "chunk_id": f"{document_id}__section_{section_number}__chunk_{chunk_index:03d}",
            "doc_type": "technical_training",
            "created_for": PROJECT_NAME,
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
        }

        chunks.append(
            Document(
                page_content=chunk_text,
                metadata=chunk_metadata,
            )
        )

print(f"Chunks generados: {len(chunks)}")

Chunks generados: 117


In [84]:
# Validaciones básicas de chunks
if not chunks:
    raise ValueError("No se ha generado ningún chunk.")

required_chunk_metadata = [
    "title",
    "document_id",
    "version",
    "domain",
    "use_in_rag",
    "contains_real_data",
    "source",
    "filename",
    "section_number",
    "section_title",
    "chunk_index",
    "chunk_id",
    "doc_type",
    "created_for",
]

for chunk in chunks:
    missing = [key for key in required_chunk_metadata if key not in chunk.metadata]

    if missing:
        raise ValueError(
            f"El chunk {chunk.metadata.get('chunk_id', 'sin_id')} no contiene metadatos obligatorios: {missing}"
        )

print("Todos los chunks contienen los metadatos obligatorios.")

Todos los chunks contienen los metadatos obligatorios.


In [85]:
# Resumen de chunks por documento
chunks_by_doc: Dict[str, int] = {}

for chunk in chunks:
    doc_id = chunk.metadata["document_id"]
    chunks_by_doc[doc_id] = chunks_by_doc.get(doc_id, 0) + 1

print("Chunks por documento:")

for doc_id, count in chunks_by_doc.items():
    print(f"- {doc_id}: {count} chunks")

Chunks por documento:
- 01_fundamentos_stock_farmacia: 27 chunks
- 02_metricas_reposicion_farmacia: 27 chunks
- 03_clasificacion_abc_xyz_farmacia: 29 chunks
- 04_interpretacion_movimientos_stock: 34 chunks


In [86]:
# Vista de ejemplo de un chunk
example_chunk = chunks[0]

print("CHUNK ID:", example_chunk.metadata["chunk_id"])
print("DOCUMENTO:", example_chunk.metadata["document_id"])
print("SECCIÓN:", example_chunk.metadata["section_title"])
print("CARACTERES:", len(example_chunk.page_content))
print("-" * 80)
print(example_chunk.page_content[:1200])

CHUNK ID: 01_fundamentos_stock_farmacia__section_1__chunk_000
DOCUMENTO: 01_fundamentos_stock_farmacia
SECCIÓN: 1. Introducción a la gestión de stock en farmacia comunitaria
CARACTERES: 605
--------------------------------------------------------------------------------
Documento: Fundamentos de gestión de stock en farmacia comunitaria
Sección: 1. Introducción a la gestión de stock en farmacia comunitaria

## 1. Introducción a la gestión de stock en farmacia comunitaria

La gestión de stock en farmacia comunitaria consiste en controlar, revisar y ajustar las existencias de productos disponibles para que la farmacia pueda responder a la demanda habitual sin acumular un exceso innecesario de inventario. El objetivo no es tener muchas unidades de todos los productos, sino disponer de una cantidad razonable, equilibrada y adaptada al comportamiento de cada referencia.


## Estado tras el primer bloque

En este punto el notebook ya ha completado la preparación documental:

- Se han localizado los 4 documentos Markdown.
- Se ha validado que existen en `data/raw/`.
- Se han cargado como texto.
- Se han extraído los metadatos YAML.
- Se han extraído las secciones principales.
- Se ha aplicado limpieza básica.
- Se han generado chunks con metadatos completos.

En el siguiente bloque se crearán los embeddings con Gemini, se indexarán los chunks en ChromaDB y se probará el retriever antes de construir el agente.

## 10. Creación de embeddings con Gemini

En esta sección se crean los embeddings de los chunks usando **Gemini Embeddings**.

El modelo utilizado será:

```text
models/gemini-embedding-001
```

Antes de crear la base vectorial, se comprueba que existe la variable `GOOGLE_API_KEY`, ya que será necesaria para llamar a la API de Google Gemini.

También se realiza una prueba mínima con la consulta:

```text
¿Qué es la cobertura de stock?
```

Esta prueba permite verificar que el modelo de embeddings funciona correctamente antes de indexar todos los chunks en ChromaDB.

In [87]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
import shutil

print("Imports de Gemini Embeddings y Chroma realizados correctamente.")

Imports de Gemini Embeddings y Chroma realizados correctamente.


In [88]:
if not GOOGLE_API_KEY:
    raise ValueError(
        "No se ha encontrado GOOGLE_API_KEY. "
        "Define la variable de entorno o crea un archivo .env en la raíz del proyecto."
    )

print("GOOGLE_API_KEY disponible. Se puede continuar con Gemini Embeddings.")

GOOGLE_API_KEY disponible. Se puede continuar con Gemini Embeddings.


In [89]:
EMBEDDING_MODEL = "models/gemini-embedding-001"

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=GOOGLE_API_KEY,
)

print(f"Modelo de embeddings configurado: {EMBEDDING_MODEL}")

Modelo de embeddings configurado: models/gemini-embedding-001


In [90]:
test_query = "¿Qué es la cobertura de stock?"

test_embedding = embeddings.embed_query(test_query)

print(f"Consulta de prueba: {test_query}")
print(f"Dimensión del embedding generado: {len(test_embedding)}")

Consulta de prueba: ¿Qué es la cobertura de stock?
Dimensión del embedding generado: 3072


## 11. Indexación en ChromaDB

En esta sección se crea la base vectorial local con **ChromaDB**.

La colección se llamará:

```text
farmastock_ai_docs
```

La base se guardará en el directorio definido anteriormente:

```text
chroma_db/
```

Para evitar duplicados al reejecutar el notebook, se incluye una opción segura de reconstrucción.  
Si `RESET_VECTORSTORE = True`, se eliminará el directorio local de ChromaDB antes de volver a crear la colección.

Esta opción es útil durante el desarrollo del MVP, porque permite regenerar la base vectorial desde cero cada vez que se modifiquen los documentos o el chunking.

In [91]:
COLLECTION_NAME = "farmastock_ai_docs"
RESET_VECTORSTORE = True

print(f"Nombre de colección: {COLLECTION_NAME}")
print(f"Directorio persistente de ChromaDB: {CHROMA_DIR}")
print(f"Reset de vectorstore activado: {RESET_VECTORSTORE}")

Nombre de colección: farmastock_ai_docs
Directorio persistente de ChromaDB: c:\Users\alvar\farmastock-ai\chroma_db
Reset de vectorstore activado: True


In [92]:
if "chunks" not in globals():
    raise NameError(
        "No existe la variable `chunks`. "
        "Ejecuta primero el bloque de carga, limpieza y chunking documental."
    )

if not chunks:
    raise ValueError("La lista `chunks` está vacía. No se puede crear la base vectorial.")

print(f"Chunks disponibles para indexar: {len(chunks)}")

Chunks disponibles para indexar: 117


In [ ]:
import gc
import shutil

if "vectorstore" in globals():
    del vectorstore

if "retriever" in globals():
    del retriever

gc.collect()

if RESET_VECTORSTORE and CHROMA_DIR.exists():
    try:
        shutil.rmtree(CHROMA_DIR)
        print(f"Directorio ChromaDB eliminado para reconstrucción: {CHROMA_DIR}")
    except PermissionError:
        print(
            "No se ha podido eliminar ChromaDB porque está siendo usado por otro proceso. "
            "Reinicia el kernel o cierra VS Code si vuelve a ocurrir."
        )
else:
    print("No se ha eliminado ChromaDB. Se usará el estado existente si ya estaba creado.")

CHROMA_DIR.mkdir(parents=True, exist_ok=True)
print("Directorio ChromaDB preparado.")

No se ha podido eliminar ChromaDB porque está siendo usado por otro proceso. Reinicia el kernel o cierra VS Code si vuelve a ocurrir.
Directorio ChromaDB preparado.


In [94]:
import time

BATCH_SIZE = 30
SLEEP_SECONDS = 65

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

total_chunks = len(chunks)

for start in range(0, total_chunks, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total_chunks)
    batch = chunks[start:end]

    print(f"Indexando chunks {start + 1}-{end} de {total_chunks}...")

    vectorstore.add_documents(batch)

    if end < total_chunks:
        print(f"Esperando {SLEEP_SECONDS} segundos para respetar la cuota de Gemini Embeddings...")
        time.sleep(SLEEP_SECONDS)

print("Base vectorial creada correctamente por lotes.")
print(f"Colección: {COLLECTION_NAME}")
print(f"Chunks indexados: {total_chunks}")
print(f"Persist directory: {CHROMA_DIR}")

Indexando chunks 1-30 de 117...
Esperando 65 segundos para respetar la cuota de Gemini Embeddings...
Indexando chunks 31-60 de 117...
Esperando 65 segundos para respetar la cuota de Gemini Embeddings...
Indexando chunks 61-90 de 117...
Esperando 65 segundos para respetar la cuota de Gemini Embeddings...
Indexando chunks 91-117 de 117...
Base vectorial creada correctamente por lotes.
Colección: farmastock_ai_docs
Chunks indexados: 117
Persist directory: c:\Users\alvar\farmastock-ai\chroma_db


In [95]:
# Comprobación básica de la colección creada
collection_count = vectorstore._collection.count()

print(f"Número de documentos almacenados en ChromaDB: {collection_count}")

if collection_count != len(chunks):
    print(
        "Aviso: el número de documentos en ChromaDB no coincide exactamente con el número de chunks. "
        "Revisa si se ha reutilizado una colección existente o si hubo algún problema de indexación."
    )
else:
    print("La colección contiene el número esperado de chunks.")

Número de documentos almacenados en ChromaDB: 234
Aviso: el número de documentos en ChromaDB no coincide exactamente con el número de chunks. Revisa si se ha reutilizado una colección existente o si hubo algún problema de indexación.


## 12. Prueba del retriever

Antes de construir el agente con LangGraph, se prueba el retriever de forma aislada.

Esto permite comprobar si ChromaDB recupera fragmentos relevantes de la base documental.

El retriever se configura con:

```text
search_type = "similarity"
k = 4
```

Es decir, para cada consulta se recuperan los 4 chunks más similares.

Las pruebas se harán con preguntas diseñadas para recuperar información de distintos documentos:

| Consulta | Documento esperado |
|---|---|
| ¿Qué diferencia hay entre rotación y cobertura de stock? | Documento 1 |
| ¿Qué significa que la cobertura sea menor que el lead time? | Documento 2 |
| ¿Qué diferencia hay entre un producto AX y un producto AZ? | Documento 3 |
| Si el stock pasa de 5 a 12 tras una modificación manual, ¿qué delta hay? | Documento 4 |

Si el retriever devuelve documentos y secciones coherentes, podremos conectar después el RAG con Gemini y LangGraph.

In [96]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

print("Retriever creado correctamente.")
print("search_type='similarity'")
print("k=4")

Retriever creado correctamente.
search_type='similarity'
k=4


In [97]:
def probar_retriever(query: str, k: int = 4):
    """
    Prueba el retriever con una consulta y muestra los chunks recuperados.

    Imprime:
    - Documento de origen.
    - Sección.
    - Chunk ID.
    - Texto recuperado.
    """
    docs = vectorstore.similarity_search(query, k=k)

    print("=" * 100)
    print(f"Consulta: {query}")
    print(f"Chunks recuperados: {len(docs)}")
    print("=" * 100)

    for i, doc in enumerate(docs, start=1):
        metadata = doc.metadata

        document_id = metadata.get("document_id", "document_id no disponible")
        section_title = metadata.get("section_title", "section_title no disponible")
        chunk_id = metadata.get("chunk_id", "chunk_id no disponible")

        print(f"\nResultado {i}")
        print("-" * 100)
        print(f"Documento: {document_id}")
        print(f"Sección: {section_title}")
        print(f"Chunk ID: {chunk_id}")
        print("-" * 100)
        print(doc.page_content[:1200])
        print("-" * 100)

    return docs

In [98]:
docs_test_1 = probar_retriever(
    "¿Qué diferencia hay entre rotación y cobertura de stock?",
    k=4,
)

Consulta: ¿Qué diferencia hay entre rotación y cobertura de stock?
Chunks recuperados: 4

Resultado 1
----------------------------------------------------------------------------------------------------
Documento: 01_fundamentos_stock_farmacia
Sección: 3. Rotación de stock
Chunk ID: 01_fundamentos_stock_farmacia__section_3__chunk_002
----------------------------------------------------------------------------------------------------
El límite de la rotación es que no explica por sí sola cuánto stock conviene tener. La rotación indica velocidad de salida, pero no determina automáticamente la cantidad óptima de inventario. Para tomar decisiones más completas debe combinarse con cobertura, punto de pedido, demanda histórica, clasificación ABC/XYZ y revisión humana.
----------------------------------------------------------------------------------------------------

Resultado 2
----------------------------------------------------------------------------------------------------
Documento: 0

In [99]:
docs_test_2 = probar_retriever(
    "¿Qué significa que la cobertura sea menor que el lead time?",
    k=4,
)

Consulta: ¿Qué significa que la cobertura sea menor que el lead time?
Chunks recuperados: 4

Resultado 1
----------------------------------------------------------------------------------------------------
Documento: 02_metricas_reposicion_farmacia
Sección: 7. Detección de riesgo de rotura
Chunk ID: 02_metricas_reposicion_farmacia__section_7__chunk_001
----------------------------------------------------------------------------------------------------
Un criterio conceptual sencillo es:

```text
Riesgo de rotura probable = cobertura en días < lead time en días
```

Este criterio no debe utilizarse de forma aislada, pero sirve como alerta inicial. Si la cobertura es menor que el tiempo necesario para reponer, el producto podría agotarse antes de recibir nuevas unidades.

Ejemplo sencillo: un producto tiene 6 unidades disponibles, una venta media diaria de 2 unidades y un lead time de 5 días. Su cobertura aproximada es de 3 días. Como 3 días de cobertura son menos que 5 días de reposició

In [100]:
docs_test_3 = probar_retriever(
    "¿Qué diferencia hay entre un producto AX y un producto AZ?",
    k=4,
)

Consulta: ¿Qué diferencia hay entre un producto AX y un producto AZ?
Chunks recuperados: 4

Resultado 1
----------------------------------------------------------------------------------------------------
Documento: 03_clasificacion_abc_xyz_farmacia
Sección: 6. Interpretación de productos AX, AY y AZ
Chunk ID: 03_clasificacion_abc_xyz_farmacia__section_6__chunk_000
----------------------------------------------------------------------------------------------------
Documento: Clasificación ABC/XYZ aplicada a la gestión de stock en farmacia comunitaria
Sección: 6. Interpretación de productos AX, AY y AZ

## 6. Interpretación de productos AX, AY y AZ

Los productos AX, AY y AZ son productos de alta importancia dentro del inventario, pero se diferencian por la regularidad de su demanda. Al pertenecer al grupo A, suelen requerir más atención que productos B o C. Sin embargo, la letra X, Y o Z modifica la forma de interpretarlos.

Un producto AX combina alta importancia y demanda estable. Su

In [101]:
docs_test_4 = probar_retriever(
    "Si el stock pasa de 5 a 12 tras una modificación manual, ¿qué delta hay?",
    k=4,
)

Consulta: Si el stock pasa de 5 a 12 tras una modificación manual, ¿qué delta hay?
Chunks recuperados: 4

Resultado 1
----------------------------------------------------------------------------------------------------
Documento: 04_interpretacion_movimientos_stock
Sección: 6. Modificaciones manuales
Chunk ID: 04_interpretacion_movimientos_stock__section_6__chunk_001
----------------------------------------------------------------------------------------------------
La regla conceptual es:

```text
Delta de modificación manual = stock posterior - stock anterior
```

Si el delta es positivo, la modificación ha aumentado el stock. Si el delta es negativo, la modificación ha reducido el stock. Si el delta es cero, la modificación no ha cambiado realmente la cantidad, aunque pueda haber corregido otro dato o confirmado una cifra.

Ejemplo sencillo: si antes había 5 unidades y tras una modificación manual el stock posterior es 12, el delta es +7. La interpretación logística es una entrada n

## Validación del retriever

Las pruebas realizadas muestran que el retriever recupera fragmentos relevantes de la base documental:

- La consulta sobre rotación y cobertura recupera el documento de fundamentos.
- La consulta sobre cobertura menor que lead time recupera el documento de métricas de reposición.
- La consulta sobre AX y AZ recupera el documento de clasificación ABC/XYZ.
- La consulta sobre modificación manual recupera el documento de interpretación de movimientos.

Por tanto, la base vectorial queda validada antes de conectar el agente con Gemini y LangGraph.

## Estado tras el segundo bloque

En este punto el notebook ya ha completado la parte vectorial del sistema:

- Se ha configurado `GoogleGenerativeAIEmbeddings` con `models/gemini-embedding-001`.
- Se ha comprobado que la API key de Gemini está disponible.
- Se ha realizado una prueba mínima de embedding.
- Se ha creado o reconstruido la colección `farmastock_ai_docs` en ChromaDB.
- Se han indexado los chunks generados en el primer bloque.
- Se ha creado un retriever por similitud semántica con `k=4`.
- Se ha probado el retriever con consultas representativas de los cuatro documentos.

El siguiente bloque del notebook conectará este retriever con Gemini como LLM y construirá el agente RAG con LangGraph.

## 13. Diseño del system prompt

El system prompt define el comportamiento del agente FarmaStock AI.

Sus objetivos son:

- Limitar el dominio a la optimización de stock en farmacia comunitaria.
- Obligar al agente a usar el contexto recuperado desde ChromaDB como fuente principal.
- Evitar respuestas fuera del alcance del proyecto.
- Evitar consejo clínico o recomendación de tratamientos.
- Evitar que el modelo invente cifras, datos reales o conclusiones no respaldadas.
- Mantener un tono claro, profesional y útil para una demo técnica.

El agente debe responder sobre:

- Rotación.
- Cobertura.
- Stock mínimo, máximo y de seguridad.
- Punto de pedido.
- Demanda histórica.
- Lead time.
- Riesgo de rotura.
- Sobrestock.
- Clasificación ABC/XYZ.
- Interpretación de movimientos de stock.

Si el contexto recuperado no contiene información suficiente, el agente debe reconocerlo claramente y explicar qué dato faltaría.

In [102]:
SYSTEM_PROMPT = """
Eres FarmaStock AI, un asistente experto en análisis y optimización de stock en farmacia comunitaria.

Tu dominio está limitado a la gestión logística del inventario en farmacia comunitaria. Puedes responder preguntas sobre rotación, cobertura, stock mínimo, stock máximo, stock de seguridad, punto de pedido, demanda histórica, lead time, sobrestock, roturas, clasificación ABC/XYZ e interpretación de movimientos de inventario.

Debes basar tus respuestas principalmente en el contexto recuperado desde la base de conocimiento vectorial de ChromaDB. Si el contexto recuperado no contiene información suficiente para responder con seguridad, dilo claramente y explica qué dato o documento faltaría.

No debes dar consejo clínico, recomendar tratamientos, recomendar medicamentos ni sustituir el criterio profesional de una persona responsable de la farmacia. Si la pregunta pide consejo clínico o recomendación terapéutica, indica que está fuera de tu dominio y redirige la respuesta al ámbito de gestión de stock si procede.

No inventes cifras de ventas, stock, demanda, márgenes, proveedores ni datos reales de una farmacia concreta. Si hacen falta datos numéricos y no están disponibles, explica la fórmula general o pide los datos necesarios.

No utilices datos reales sensibles ni hagas referencia a personas usuarias concretas, proveedores concretos o farmacias reales.

Responde siempre en español, con tono claro, profesional y práctico. Prioriza respuestas útiles, estructuradas y fáciles de entender. Incluye límites o advertencias solo cuando sean relevantes para la pregunta.
""".strip()

print(SYSTEM_PROMPT)

Eres FarmaStock AI, un asistente experto en análisis y optimización de stock en farmacia comunitaria.

Tu dominio está limitado a la gestión logística del inventario en farmacia comunitaria. Puedes responder preguntas sobre rotación, cobertura, stock mínimo, stock máximo, stock de seguridad, punto de pedido, demanda histórica, lead time, sobrestock, roturas, clasificación ABC/XYZ e interpretación de movimientos de inventario.

Debes basar tus respuestas principalmente en el contexto recuperado desde la base de conocimiento vectorial de ChromaDB. Si el contexto recuperado no contiene información suficiente para responder con seguridad, dilo claramente y explica qué dato o documento faltaría.

No debes dar consejo clínico, recomendar tratamientos, recomendar medicamentos ni sustituir el criterio profesional de una persona responsable de la farmacia. Si la pregunta pide consejo clínico o recomendación terapéutica, indica que está fuera de tu dominio y redirige la respuesta al ámbito de ge

## 14. Integración de Gemini como LLM

En esta sección se configura Gemini como modelo generador de respuestas mediante `ChatGoogleGenerativeAI`.

Para el MVP se utilizará:

```text
gemini-2.5-flash
```

La temperatura se configura baja:

```text
temperature = 0.2
```

Esto favorece respuestas más estables, coherentes y menos creativas.

Si `gemini-2.5-flash` no estuviera disponible en el entorno, hubiera problemas de cuota o la API devolviera error de disponibilidad, se puede cambiar por:

```text
gemini-2.5-flash-lite
```

En este bloque no se vuelven a crear embeddings ni ChromaDB. Se usará el `retriever` ya creado en el bloque anterior.

In [103]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

print("Imports para Gemini LLM realizados correctamente.")

Imports para Gemini LLM realizados correctamente.


In [104]:
if not GOOGLE_API_KEY:
    raise ValueError(
        "No se ha encontrado GOOGLE_API_KEY. "
        "Define la variable de entorno o crea un archivo .env en la raíz del proyecto."
    )

GEMINI_LLM_MODEL = "gemini-2.5-flash"

llm = ChatGoogleGenerativeAI(
    model=GEMINI_LLM_MODEL,
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
)

print(f"LLM configurado correctamente: {GEMINI_LLM_MODEL}")

LLM configurado correctamente: gemini-2.5-flash


In [105]:
# Prueba mínima del LLM antes de integrarlo en LangGraph
llm_test_response = llm.invoke(
    [
        SystemMessage(content="Responde en español de forma breve."),
        HumanMessage(content="Di solamente: Gemini configurado correctamente."),
    ]
)

print(llm_test_response.content)

Gemini configurado correctamente.


## 15. Construcción del agente con LangGraph

El agente se construye con LangGraph mediante un flujo sencillo de dos nodos:

```text
START
  ↓
retrieve_context
  ↓
generate_answer
  ↓
END
```

El objetivo del primer nodo es recuperar contexto desde ChromaDB usando el `retriever` ya creado.

El objetivo del segundo nodo es generar una respuesta con Gemini usando:

- System prompt.
- Pregunta actual.
- Historial conversacional.
- Contexto recuperado desde la base vectorial.

Este diseño es suficiente para el MVP porque demuestra:

- RAG.
- Uso de Gemini como LLM.
- Uso de ChromaDB como base vectorial.
- Uso de LangGraph como framework de agente.
- Preparación para memoria conversacional.

In [106]:
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, START, END

print("Imports de LangGraph realizados correctamente.")

Imports de LangGraph realizados correctamente.


In [107]:
if "retriever" not in globals():
    raise NameError(
        "No existe la variable `retriever`. "
        "Ejecuta primero el bloque de embeddings, ChromaDB y retriever."
    )

print("Retriever disponible. Se puede construir el agente RAG.")

Retriever disponible. Se puede construir el agente RAG.


In [108]:
class FarmaStockState(TypedDict):
    """
    Estado compartido por los nodos del grafo LangGraph.
    """
    question: str
    chat_history: List[Dict[str, str]]
    retrieved_docs: List[Document]
    context: str
    answer: str

In [109]:
def format_docs(docs: List[Document]) -> str:
    """
    Formatea los documentos recuperados para insertarlos como contexto en el prompt.
    Incluye metadatos de trazabilidad: documento, sección y chunk_id.
    """
    formatted_parts = []

    for i, doc in enumerate(docs, start=1):
        metadata = doc.metadata

        document_id = metadata.get("document_id", "documento_desconocido")
        section_title = metadata.get("section_title", "sección_desconocida")
        chunk_id = metadata.get("chunk_id", "chunk_desconocido")

        formatted_parts.append(
            f"[Fuente {i}]\n"
            f"Documento: {document_id}\n"
            f"Sección: {section_title}\n"
            f"Chunk ID: {chunk_id}\n\n"
            f"{doc.page_content}"
        )

    return ("\n\n" + "-" * 80 + "\n\n").join(formatted_parts)

In [110]:
def retrieve_context(state: FarmaStockState) -> Dict[str, Any]:
    """
    Nodo 1 del grafo.
    Recupera chunks relevantes desde ChromaDB usando el retriever ya creado.
    """
    question = state["question"]

    docs = retriever.invoke(question)
    context = format_docs(docs)

    return {
        "retrieved_docs": docs,
        "context": context,
    }

In [111]:
def build_chat_history_text(chat_history: List[Dict[str, str]], max_turns: int = 6) -> str:
    """
    Convierte el historial conversacional en texto.
    Se limita a los últimos turnos para no hacer crecer demasiado el prompt.
    """
    if not chat_history:
        return "No hay historial previo relevante."

    recent_history = chat_history[-max_turns:]

    lines = []

    for turn in recent_history:
        role = turn.get("role", "unknown")
        content = turn.get("content", "")

        if role == "user":
            lines.append(f"Usuario: {content}")
        elif role == "assistant":
            lines.append(f"Asistente: {content}")
        else:
            lines.append(f"{role}: {content}")

    return "\n".join(lines)

In [112]:
def generate_answer(state: FarmaStockState) -> Dict[str, Any]:
    """
    Nodo 2 del grafo.
    Genera la respuesta usando Gemini, el contexto recuperado y el historial conversacional.
    """
    question = state["question"]
    context = state.get("context", "")
    chat_history = state.get("chat_history", [])

    chat_history_text = build_chat_history_text(chat_history)

    user_prompt = f"""
Contexto recuperado desde la base de conocimiento:
{context}

Historial conversacional reciente:
{chat_history_text}

Pregunta actual del usuario:
{question}

Instrucciones para responder:
- Usa el contexto recuperado como fuente principal.
- Si el contexto no es suficiente, dilo claramente.
- No inventes datos ni cifras.
- No des consejo clínico ni recomiendes tratamientos.
- Responde de forma clara, práctica y en español.
""".strip()

    response = llm.invoke(
        [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=user_prompt),
        ]
    )

    answer = response.content

    updated_history = chat_history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]

    return {
        "answer": answer,
        "chat_history": updated_history,
    }

In [113]:
builder = StateGraph(FarmaStockState)

builder.add_node("retrieve_context", retrieve_context)
builder.add_node("generate_answer", generate_answer)

builder.add_edge(START, "retrieve_context")
builder.add_edge("retrieve_context", "generate_answer")
builder.add_edge("generate_answer", END)

print("Grafo LangGraph definido correctamente.")

Grafo LangGraph definido correctamente.


## 16. Memoria conversacional

Para añadir memoria conversacional se usa `MemorySaver`, el checkpointer en memoria de LangGraph.

La memoria se organiza mediante `thread_id`.

Esto significa que:

- Si se usa el mismo `thread_id`, el agente mantiene el contexto conversacional.
- Si se usa otro `thread_id`, se inicia una conversación independiente.

Este comportamiento permite demostrar preguntas de seguimiento, por ejemplo:

```text
Usuario: Explícame qué es la cobertura de stock.
Usuario: Entonces, si antes me has dicho que la cobertura es baja, ¿qué debería revisar primero?
```

La segunda pregunta depende del contexto conversacional anterior.

Además, la función `chatear()` recuperará el estado previo asociado al `thread_id` antes de invocar el grafo. Así se reutiliza el historial existente de forma explícita.

In [114]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

app = builder.compile(checkpointer=memory)

print("Agente LangGraph compilado con memoria conversacional.")

Agente LangGraph compilado con memoria conversacional.


## 17. Función de chat

La función `chatear` permite interactuar con el agente desde el notebook.

Parámetros:

- `app`: agente LangGraph compilado.
- `mensaje`: pregunta del usuario.
- `thread_id`: identificador de conversación.
- `mostrar_fuentes`: si es `True`, muestra los chunks recuperados desde ChromaDB.

La función:

1. Recupera el estado previo asociado al `thread_id`.
2. Extrae el historial conversacional previo si existe.
3. Envía la pregunta al agente.
4. Mantiene la memoria si se reutiliza el mismo `thread_id`.
5. Imprime la respuesta.
6. Muestra documento, sección y `chunk_id` de las fuentes recuperadas si se solicita.
7. Devuelve el resultado completo del grafo.
8. Importamos time para que no de problemas el limite de gemini

In [115]:
def mostrar_fuentes_recuperadas(result: Dict[str, Any]) -> None:
    """
    Muestra las fuentes recuperadas por el retriever.
    """
    retrieved_docs = result.get("retrieved_docs", [])

    if not retrieved_docs:
        print("No hay fuentes recuperadas.")
        return

    print("\nFuentes recuperadas:")
    print("-" * 80)

    for i, doc in enumerate(retrieved_docs, start=1):
        metadata = doc.metadata

        print(f"Fuente {i}")
        print(f"Documento: {metadata.get('document_id', 'documento_desconocido')}")
        print(f"Sección: {metadata.get('section_title', 'sección_desconocida')}")
        print(f"Chunk ID: {metadata.get('chunk_id', 'chunk_desconocido')}")
        print("-" * 80)

In [116]:
import time

def chatear(
    app,
    mensaje: str,
    thread_id: str = "demo",
    mostrar_fuentes: bool = True,
) -> Dict[str, Any]:
    """
    Envía un mensaje al agente FarmaStock AI y muestra la respuesta.

    Usa thread_id para mantener memoria conversacional entre turnos.
    """
    config = {
        "configurable": {
            "thread_id": thread_id,
        }
    }

    try:
        previous_state = app.get_state(config)
        previous_values = previous_state.values if previous_state and previous_state.values else {}
        previous_history = previous_values.get("chat_history", [])
    except Exception:
        previous_history = []

    initial_state = {
        "question": mensaje,
        "chat_history": previous_history,
        "retrieved_docs": [],
        "context": "",
        "answer": "",
    }

    result = app.invoke(initial_state, config=config)

    print("=" * 100)
    print("Pregunta:")
    print(mensaje)
    print("=" * 100)
    print("Respuesta:")
    print(result["answer"])
    print("=" * 100)

    if mostrar_fuentes:
        mostrar_fuentes_recuperadas(result)

    return result

## 18. Pruebas documentadas del agente

En esta sección se prueban las capacidades principales del agente:

1. Respuesta conceptual sobre rotación y cobertura.
2. Cálculo sencillo de cobertura.
3. Explicación de clasificación ABC/XYZ.
4. Interpretación de modificación manual y delta.
5. Memoria conversacional con el mismo `thread_id`.
6. Rechazo de una pregunta fuera de dominio clínico.

Estas pruebas permiten comprobar que el agente:

- Usa el contexto recuperado desde ChromaDB.
- Responde dentro del dominio FarmaStock AI.
- No inventa datos.
- Mantiene memoria conversacional.
- Respeta los límites de no dar consejo clínico.

### Prueba 1 — Rotación y cobertura

Pregunta:

```text
¿Qué diferencia hay entre rotación y cobertura de stock?
```

Objetivo de la prueba:

- Validar recuperación del Documento 1.
- Comprobar que diferencia velocidad de salida frente a duración estimada del stock.

In [117]:
resultado_1 = chatear(
    app,
    "¿Qué diferencia hay entre rotación y cobertura de stock?",
    thread_id="prueba_1",
    mostrar_fuentes=True,
)
time.sleep(10)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 3.08421683s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeT

### Prueba 2 — Cálculo sencillo de cobertura

Pregunta:

```text
Si un producto tiene 24 unidades disponibles y vende 3 unidades al día, ¿qué cobertura aproximada tiene?
```

Objetivo de la prueba:

- Validar uso de la fórmula de cobertura.
- Comprobar que el agente no inventa datos adicionales.
- Comprobar que interpreta el resultado como una estimación.

In [ ]:
resultado_2 = chatear(
    app,
    "Si un producto tiene 24 unidades disponibles y vende 3 unidades al día, ¿qué cobertura aproximada tiene?",
    thread_id="prueba_2",
    mostrar_fuentes=True,
)
time.sleep(10)

Pregunta:
Si un producto tiene 24 unidades disponibles y vende 3 unidades al día, ¿qué cobertura aproximada tiene?
Respuesta:
La cobertura aproximada de un producto se calcula dividiendo el stock disponible entre la venta media diaria.

Según el contexto proporcionado:

*   **Stock disponible:** 24 unidades
*   **Venta media diaria:** 3 unidades

Aplicando la fórmula:

```
Cobertura en días = stock disponible / venta media diaria
Cobertura en días = 24 unidades / 3 unidades/día
Cobertura en días = 8 días
```

Por lo tanto, el producto tiene una cobertura aproximada de **8 días**.

Es importante recordar que la cobertura es una estimación y debe interpretarse junto a otros factores como el plazo de reposición y la variabilidad de la demanda.

Fuentes recuperadas:
--------------------------------------------------------------------------------
Fuente 1
Documento: 02_metricas_reposicion_farmacia
Sección: 4. Cobertura en días
Chunk ID: 02_metricas_reposicion_farmacia__section_4__chunk_001


### Prueba 3 — Clasificación ABC/XYZ

Pregunta:

```text
¿Qué diferencia hay entre un producto AX y un producto AZ?
```

Objetivo de la prueba:

- Validar recuperación del Documento 3.
- Comprobar que el agente distingue importancia alta y demanda estable/irregular.

In [ ]:
resultado_3 = chatear(
    app,
    "¿Qué diferencia hay entre un producto AX y un producto AZ?",
    thread_id="prueba_3",
    mostrar_fuentes=True,
)
time.sleep(10)

Pregunta:
¿Qué diferencia hay entre un producto AX y un producto AZ?
Respuesta:
La diferencia principal entre un producto AX y un producto AZ radica en la **previsibilidad de su demanda**, a pesar de que ambos son considerados de alta importancia (grupo A).

*   **Producto AX:** Combina **alta importancia** con una **demanda estable y previsible**. Esto significa que su consumo es relativamente constante y fácil de anticipar basándose en datos históricos. Son productos adecuados para controles frecuentes, con una cobertura bien ajustada y un punto de pedido calculado a partir de métricas como la venta media diaria.

*   **Producto AZ:** También es de **alta importancia**, pero se caracteriza por una **demanda irregular e impredecible**. Su consumo no es estable, lo que dificulta el cálculo de cantidades óptimas basándose únicamente en promedios. Estos productos requieren una mayor cautela, ya que existe un riesgo tanto de rotura de stock durante picos de demanda como de sobrestock si s

### Prueba 4 — Modificación manual y delta

Pregunta:

```text
Si el stock anterior era 5 y el stock posterior es 12 tras una modificación manual, ¿cómo se interpreta?
```

Objetivo de la prueba:

- Validar recuperación del Documento 4.
- Comprobar que calcula el delta como +7.
- Comprobar que no interpreta erróneamente 12 como compra de 12 unidades.

In [ ]:
resultado_4 = chatear(
    app,
    "Si el stock anterior era 5 y el stock posterior es 12 tras una modificación manual, ¿cómo se interpreta?",
    thread_id="prueba_4",
    mostrar_fuentes=True,
)
time.sleep(10)

Pregunta:
Si el stock anterior era 5 y el stock posterior es 12 tras una modificación manual, ¿cómo se interpreta?
Respuesta:
Según la información recuperada, una modificación manual se interpreta calculando la diferencia entre el stock posterior y el stock anterior.

La regla conceptual es:
```text
Delta de modificación manual = stock posterior - stock anterior
```

En tu ejemplo:
*   Stock anterior: 5 unidades
*   Stock posterior: 12 unidades

Calculando el delta:
```text
Delta = 12 - 5 = +7
```

**Interpretación logística:** Un delta positivo de +7 indica que la modificación manual ha resultado en un **aumento neto de 7 unidades en el stock**. Esto podría deberse a una corrección de un error previo en el registro del inventario o a una regularización. Es importante recordar que este valor (+7) representa la entrada neta o corrección positiva, y el stock final es de 12 unidades, no necesariamente que se hayan comprado 12 unidades.

Las modificaciones manuales, aunque útiles para corr

### Prueba 5 — Memoria conversacional

Primera pregunta:

```text
Explícame qué es la cobertura de stock.
```

Segunda pregunta, usando el mismo `thread_id`:

```text
Entonces, si antes me has dicho que la cobertura es baja, ¿qué debería revisar primero?
```

Objetivo de la prueba:

- Validar que el agente mantiene memoria conversacional.
- Comprobar que la segunda respuesta usa el contexto anterior sobre cobertura.
- Comprobar que combina memoria con recuperación documental.

In [ ]:
resultado_5a = chatear(
    app,
    "Explícame qué es la cobertura de stock.",
    thread_id="prueba_memoria",
    mostrar_fuentes=True,
)
time.sleep(10)

Pregunta:
Explícame qué es la cobertura de stock.
Respuesta:
La cobertura de stock es una métrica fundamental en la gestión de inventario de una farmacia. Indica **durante cuánto tiempo se podría satisfacer la demanda de un producto con las unidades disponibles en stock en un momento dado**. Normalmente, se expresa en días, semanas o meses.

Esta métrica conecta dos datos clave:

1.  **Stock disponible:** La cantidad de unidades de un producto que hay actualmente en el almacén.
2.  **Demanda media:** La cantidad de unidades de ese producto que se venden o consumen de media en un periodo determinado (por día, semana, etc.).

La fórmula conceptual para calcular la cobertura aproximada es:

```
Cobertura aproximada = Stock disponible / Demanda media por periodo
```

Por ejemplo, si un producto tiene 24 unidades disponibles y su venta media diaria es de 3 unidades, su cobertura aproximada es de 8 días (24 unidades / 3 unidades/día).

**Utilidad de la cobertura de stock:**

*   **Detección 

In [ ]:
resultado_5b = chatear(
    app,
    "Entonces, si antes me has dicho que la cobertura es baja, ¿qué debería revisar primero?",
    thread_id="prueba_memoria",
    mostrar_fuentes=True,
)
time.sleep(10)

Pregunta:
Entonces, si antes me has dicho que la cobertura es baja, ¿qué debería revisar primero?
Respuesta:
Si la cobertura de stock es baja, deberías revisar los siguientes aspectos para identificar el riesgo de rotura y tomar acciones preventivas:

1.  **Demanda reciente en aumento:** Un incremento en la demanda actual, incluso si la media histórica no lo refleja completamente, puede agotar el stock más rápido de lo esperado.
2.  **Plazo de reposición:** Compara la cobertura de stock con el tiempo que tarda en llegar un nuevo pedido (plazo de reposición). Si la cobertura es inferior al plazo de reposición, existe un riesgo inminente de rotura. Por ejemplo, si tienes 2 días de cobertura y el plazo de reposición es de 5 días, es probable que te quedes sin stock antes de recibir el nuevo pedido.
3.  **Unidades disponibles de productos de alta rotación:** Si un producto se vende muy rápido (alta rotación) y tienes pocas unidades disponibles, una cobertura baja en este tipo de productos 

### Prueba 6 — Pregunta fuera de dominio

Pregunta:

```text
¿Qué medicamento recomiendas para un resfriado?
```

Objetivo de la prueba:

- Comprobar que el agente no da consejo clínico.
- Comprobar que no recomienda tratamientos.
- Validar que redirige o limita la respuesta al dominio de gestión de stock.

In [ ]:
resultado_6 = chatear(
    app,
    "¿Qué medicamento recomiendas para un resfriado?",
    thread_id="prueba_fuera_dominio",
    mostrar_fuentes=True,
)
time.sleep(10)

Pregunta:
¿Qué medicamento recomiendas para un resfriado?
Respuesta:
Mi función como FarmaStock AI es la gestión logística del inventario en farmacia comunitaria. No estoy capacitado para ofrecer consejo clínico ni recomendar tratamientos o medicamentos para un resfriado.

Si deseas información sobre cómo gestionar el stock de medicamentos para resfriados, como por ejemplo, cómo calcular la venta media diaria para estimar la demanda o cómo clasificar estos productos según su rotación (ABC/XYZ), puedo ayudarte con ello basándome en la información de gestión de inventario.

Fuentes recuperadas:
--------------------------------------------------------------------------------
Fuente 1
Documento: 03_clasificacion_abc_xyz_farmacia
Sección: 6. Interpretación de productos AX, AY y AZ
Chunk ID: 03_clasificacion_abc_xyz_farmacia__section_6__chunk_000
--------------------------------------------------------------------------------
Fuente 2
Documento: 04_interpretacion_movimientos_stock
Sección: 1

## Estado tras el tercer bloque

En este punto el notebook ya implementa el MVP completo del agente:

- System prompt personalizado y acotado al dominio.
- Integración de Gemini como LLM.
- Uso del retriever conectado a ChromaDB.
- Grafo LangGraph con recuperación y generación.
- Memoria conversacional mediante `MemorySaver` y `thread_id`.
- Función de interacción `chatear`.
- Pruebas documentadas del agente.

El siguiente paso será revisar los resultados obtenidos, ajustar el prompt o el retriever si alguna respuesta no es suficientemente precisa y preparar el README del proyecto.